# 15. The Automated Guided Vehicle Traffic Management Problem

## Tier 2 — Priority-Based Heuristic Algorithms

This notebook develops **fast heuristic algorithms** for AGV traffic management that can handle larger instances in real-time.

### Learning goals

- Understand how **priority-based scheduling** works for AGV coordination
- See how **conflict detection and resolution** can be done efficiently
- Learn how **greedy approaches** with look-ahead can provide good solutions quickly

### What this notebook outputs

- Multiple heuristic algorithms with different strategies
- Performance comparison with the MIP baseline
- Scalability analysis for larger instances

### Why heuristics matter

While MIP gives optimal solutions, heuristics provide **fast, scalable alternatives** suitable for real-time operations in large terminals where computational time is limited.

In [ ]:
# Environment check (no installs here)
#
# Best practice for classes: preinstall dependencies in the Docker/JupyterHub image.
# If you're running locally, install packages once in your environment.

try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib.patches as patches
    from itertools import product, combinations
    from dataclasses import dataclass, field
    from typing import List, Tuple, Dict, Optional, Set
    from collections import defaultdict, deque
    import heapq
    import time
    import random
except ImportError as e:
    raise ImportError(
        "Missing dependency. Install: numpy, pandas, matplotlib. "
        "If you use the provided JupyterHub Docker image, these should already be installed."
    ) from e

print("Dependencies imported successfully.")

## Extended instance (8 AGVs, 12 nodes)

For Tier-2, we use a **larger instance** to demonstrate heuristic scalability:

- **12 nodes** (larger terminal layout)
- **20 directed edges** (more complex road network)
- **8 AGVs** (double the complexity)
- **Time horizon** of 30 time units

### Enhanced time model

- Variable travel times (1-4 time units)
- **Dynamic conflict resolution** with waiting strategies
- **Priority-based right-of-way** rules

In [ ]:
# ----------------------------
# Enhanced data models for heuristics
# ----------------------------
@dataclass(frozen=True)
class Node:
    id: int
    x: float
    y: float
    is_intersection: bool = True
    capacity: int = 1  # can handle multiple AGVs in larger terminals

@dataclass(frozen=True)
class Edge:
    from_node: int
    to_node: int
    travel_time: int
    capacity: int = 1  # number of AGVs that can use simultaneously
    bidirectional: bool = False

@dataclass(frozen=True)
class AGV:
    id: int
    origin: int
    destination: int
    priority: int = 1
    max_speed: float = 1.0  # relative speed factor

@dataclass
class AGVState:
    agv_id: int
    current_node: int
    target_node: int
    current_time: int
    path: List[int] = field(default_factory=list)
    waiting: bool = False
    completed: bool = False

# ----------------------------
# Larger terminal layout: 12 nodes in a 3x4 grid
# ----------------------------
# Layout:
#   1  --  2  --  3  --  4
#   |      |      |      |
#   5  --  6  --  7  --  8
#   |      |      |      |
#   9  -- 10  -- 11  -- 12
#
nodes = [
    Node(1, 0.0, 2.0), Node(2, 1.5, 2.0), Node(3, 3.0, 2.0), Node(4, 4.5, 2.0),
    Node(5, 0.0, 1.0), Node(6, 1.5, 1.0), Node(7, 3.0, 1.0), Node(8, 4.5, 1.0),
    Node(9, 0.0, 0.0), Node(10, 1.5, 0.0), Node(11, 3.0, 0.0), Node(12, 4.5, 0.0)
]

# More complex road network
edges = [
    # Horizontal edges (top row)
    Edge(1, 2, 2), Edge(2, 1, 2), Edge(2, 3, 2), Edge(3, 2, 2),
    Edge(3, 4, 2), Edge(4, 3, 2),
    # Horizontal edges (middle row)
    Edge(5, 6, 1), Edge(6, 5, 1), Edge(6, 7, 1), Edge(7, 6, 1),
    Edge(7, 8, 1), Edge(8, 7, 1),
    # Horizontal edges (bottom row)
    Edge(9, 10, 1), Edge(10, 9, 1), Edge(10, 11, 1), Edge(11, 10, 1),
    Edge(11, 12, 1), Edge(12, 11, 1),
    # Vertical edges
    Edge(1, 5, 1), Edge(5, 1, 1), Edge(2, 6, 1), Edge(6, 2, 1),
    Edge(3, 7, 1), Edge(7, 3, 1), Edge(4, 8, 2), Edge(8, 4, 2),
    Edge(5, 9, 1), Edge(9, 5, 1), Edge(6, 10, 1), Edge(10, 6, 1),
    Edge(7, 11, 1), Edge(11, 7, 1), Edge(8, 12, 2), Edge(12, 8, 2)
]

# More AGVs with varied priorities
agvs = [
    AGV(1, 1, 12, 1, 1.0),  # high priority, normal speed
    AGV(2, 4, 9, 2, 1.2),  # medium priority, fast
    AGV(3, 2, 11, 3, 0.8),  # low priority, slow
    AGV(4, 5, 8, 2, 1.0),  # medium priority, normal speed
    AGV(5, 10, 3, 1, 1.1),  # high priority, fast
    AGV(6, 7, 1, 3, 0.9),  # low priority, slow
    AGV(7, 12, 6, 2, 1.0),  # medium priority, normal speed
    AGV(8, 9, 4, 1, 1.3),  # high priority, very fast
]

# ----------------------------
# Problem parameters
# ----------------------------
TIME_HORIZON = 30
SAFETY_HEADWAY = 1
NODE_CAPACITY = 1

# ----------------------------
# Helper data structures
# ----------------------------
node_lookup = {n.id: n for n in nodes}
edge_lookup = {(e.from_node, e.to_node): e for e in edges}
agv_lookup = {a.id: a for a in agvs}

outgoing_edges = {n.id: [e for e in edges if e.from_node == n.id] for n in nodes}
incoming_edges = {n.id: [e for e in edges if e.to_node == n.id] for n in nodes}

print(f"Extended terminal: {len(nodes)} nodes, {len(edges)} edges, {len(agvs)} AGVs")
print(f"Time horizon: {TIME_HORIZON} units")
print(f"AGV priorities: {[f'AGV{a.id}: {a.priority}' for a in agvs]}")

## Step 1 — Path planning foundation

Before scheduling AGVs, we need **efficient path planning**. We'll implement:

- **Shortest path algorithm** (Dijkstra's algorithm)
- **Alternative path generation** for conflict resolution
- **Path cost calculation** considering distance and congestion

In [ ]:
class PathPlanner:
    """Efficient path planning for AGVs"""
    
    def __init__(self, nodes: List[Node], edges: List[Edge]):
        self.nodes = nodes
        self.edges = edges
        self.node_lookup = {n.id: n for n in nodes}
        self.edge_lookup = {(e.from_node, e.to_node): e for e in edges}
        self.adjacency = self._build_adjacency()
    
    def _build_adjacency(self) -> Dict[int, List[Tuple[int, int]]]:
        """Build adjacency list for path finding"""
        adj = defaultdict(list)
        for edge in self.edges:
            adj[edge.from_node].append((edge.to_node, edge.travel_time))
        return dict(adj)
    
    def dijkstra(self, start: int, end: int) -> Tuple[List[int], int]:
        """Find shortest path using Dijkstra's algorithm"""
        distances = {node.id: float('inf') for node in self.nodes}
        distances[start] = 0
        previous = {}
        
        # Priority queue: (distance, node)
        pq = [(0, start)]
        visited = set()
        
        while pq:
            current_dist, current_node = heapq.heappop(pq)
            
            if current_node in visited:
                continue
            
            visited.add(current_node)
            
            if current_node == end:
                break
            
            for neighbor, travel_time in self.adjacency.get(current_node, []):
                if neighbor in visited:
                    continue
                
                distance = current_dist + travel_time
                
                if distance < distances[neighbor]:
                    distances[neighbor] = distance
                    previous[neighbor] = current_node
                    heapq.heappush(pq, (distance, neighbor))
        
        # Reconstruct path
        if end not in previous and start != end:
            return [], float('inf')  # No path found
        
        path = []
        current = end
        while current is not None:
            path.append(current)
            current = previous.get(current)
        
        path.reverse()
        return path, distances[end]
    
    def find_k_shortest_paths(self, start: int, end: int, k: int = 3) -> List[Tuple[List[int], int]]:
        """Find k shortest paths for alternative routing"""
        # Simple implementation: find shortest path, then remove edges and find alternatives
        paths = []
        
        # First, find the shortest path
        shortest_path, shortest_cost = self.dijkstra(start, end)
        if shortest_path:
            paths.append((shortest_path, shortest_cost))
        
        # Find alternative paths by removing edges from the shortest path
        for i in range(len(shortest_path) - 1):
            from_node, to_node = shortest_path[i], shortest_path[i + 1]
            
            # Temporarily remove this edge
            if (from_node, to_node) in self.edge_lookup:
                original_edges = self.edges.copy()
                self.edges = [e for e in self.edges if not (e.from_node == from_node and e.to_node == to_node)]
                self.adjacency = self._build_adjacency()
                
                # Find alternative path
                alt_path, alt_cost = self.dijkstra(start, end)
                if alt_path and alt_path != shortest_path:
                    paths.append((alt_path, alt_cost))
                
                # Restore edges
                self.edges = original_edges
                self.adjacency = self._build_adjacency()
        
        # Sort by cost and return top k
        paths.sort(key=lambda x: x[1])
        return paths[:k]

# Initialize path planner
path_planner = PathPlanner(nodes, edges)

# Test path planning
print("=== PATH PLANNING TEST ===")
for agv in agvs[:3]:  # Test first 3 AGVs
    path, cost = path_planner.dijkstra(agv.origin, agv.destination)
    print(f"AGV {agv.id}: {agv.origin} → {agv.destination}, cost = {cost}, path = {path}")

## Step 2 — Conflict detection system

Effective conflict detection is crucial for safe AGV operations:

- **Node conflicts**: Multiple AGVs at same node simultaneously
- **Edge conflicts**: Head-on collisions and following conflicts
- **Temporal conflicts**: Time-based safety violations

In [ ]:
class ConflictDetector:
    """Detect and classify conflicts between AGVs"""
    
    def __init__(self, safety_headway: int = 1):
        self.safety_headway = safety_headway
    
    def detect_node_conflicts(self, agv_states: List[AGVState]) -> List[Tuple[int, int, int]]:
        """Detect node conflicts: (time, node, [agv_ids])"""
        conflicts = []
        node_occupancy = defaultdict(list)  # (time, node) -> [agv_ids]
        
        for state in agv_states:
            if state.completed:
                continue
            
            # Check current position
            node_occupancy[(state.current_time, state.current_node)].append(state.agv_id)
            
            # Check planned path
            for i, node in enumerate(state.path):
                time = state.current_time + i
                node_occupancy[(time, node)].append(state.agv_id)
        
        # Find conflicts (more than 1 AGV at same node and time)
        for (time, node), agv_list in node_occupancy.items():
            if len(agv_list) > 1:
                conflicts.append((time, node, agv_list))
        
        return conflicts
    
    def detect_edge_conflicts(self, agv_states: List[AGVState]) -> List[Tuple[int, int, int, List[int]]]:
        """Detect edge conflicts: (time, from_node, to_node, [agv_ids])"""
        conflicts = []
        edge_usage = defaultdict(list)  # (time, from_node, to_node) -> [agv_ids]
        
        for state in agv_states:
            if state.completed or len(state.path) < 2:
                continue
            
            # Check edge usage along the path
            for i in range(len(state.path) - 1):
                from_node = state.path[i]
                to_node = state.path[i + 1]
                time = state.current_time + i
                
                edge_usage[(time, from_node, to_node)].append(state.agv_id)
                
                # Check for opposite direction (head-on conflict)
                opposite_key = (time, to_node, from_node)
                if opposite_key in edge_usage:
                    conflicts.append((time, from_node, to_node, 
                                   edge_usage[(time, from_node, to_node)] + edge_usage[opposite_key]))
        
        # Check for following conflicts (same direction, too close)
        for (time, from_node, to_node), agv_list in edge_usage.items():
            if len(agv_list) > 1:
                conflicts.append((time, from_node, to_node, agv_list))
        
        return conflicts
    
    def detect_all_conflicts(self, agv_states: List[AGVState]) -> Dict[str, List]:
        """Detect all types of conflicts"""
        return {
            'node_conflicts': self.detect_node_conflicts(agv_states),
            'edge_conflicts': self.detect_edge_conflicts(agv_states)
        }

# Test conflict detection
conflict_detector = ConflictDetector(SAFETY_HEADWAY)

print("=== CONFLICT DETECTION TEST ===")
# Create test AGV states
test_states = [
    AGVState(1, 1, 6, 0, [1, 5, 6]),
    AGVState(2, 2, 7, 0, [2, 6, 7]),
    AGVState(3, 5, 8, 1, [5, 6, 7, 8])
]

conflicts = conflict_detector.detect_all_conflicts(test_states)
print(f"Node conflicts: {len(conflicts['node_conflicts'])}")
print(f"Edge conflicts: {len(conflicts['edge_conflicts'])}")

for conflict in conflicts['node_conflicts'][:3]:
    print(f"  Node conflict at time {conflict[0]}, node {conflict[1]}: AGVs {conflict[2]}")

## Step 3 — Priority-based scheduling heuristic

Our first heuristic uses **priority-based scheduling** with conflict resolution:

- **Priority ordering**: Higher priority AGVs scheduled first
- **Greedy path assignment**: Shortest path for each AGV
- **Conflict resolution**: Waiting and rerouting strategies

In [ ]:
class PriorityScheduler:
    """Priority-based AGV scheduling heuristic"""
    
    def __init__(self, nodes: List[Node], edges: List[Edge], path_planner: PathPlanner, 
                 conflict_detector: ConflictDetector):
        self.nodes = nodes
        self.edges = edges
        self.path_planner = path_planner
        self.conflict_detector = conflict_detector
        self.node_lookup = {n.id: n for n in nodes}
    
    def schedule_agvs(self, agvs: List[AGV], time_horizon: int) -> Dict[int, AGVState]:
        """Schedule AGVs using priority-based approach"""
        # Sort AGVs by priority (lower number = higher priority)
        sorted_agvs = sorted(agvs, key=lambda a: a.priority)
        
        agv_states = {}
        occupied_nodes = set()  # Nodes occupied at current time
        occupied_edges = {}    # (from, to) -> (agv_id, end_time)
        
        for agv in sorted_agvs:
            # Find shortest path
            path, path_cost = self.path_planner.dijkstra(agv.origin, agv.destination)
            
            if not path:
                print(f"Warning: No path found for AGV {agv.id}")
                continue
            
            # Try to schedule with conflict resolution
            scheduled_path = self._resolve_conflicts(agv, path, occupied_nodes, occupied_edges)
            
            # Create AGV state
            state = AGVState(
                agv_id=agv.id,
                current_node=agv.origin,
                target_node=agv.destination,
                current_time=0,
                path=scheduled_path[1:],  # Exclude current node
                waiting=False,
                completed=False
            )
            
            agv_states[agv.id] = state
            
            # Update occupancy
            occupied_nodes.add(agv.origin)
            
            # Reserve edges along the path
            for i in range(len(scheduled_path) - 1):
                from_node = scheduled_path[i]
                to_node = scheduled_path[i + 1]
                edge_key = (from_node, to_node)
                travel_time = self.path_planner.edge_lookup.get(edge_key, 
                                                              type('Edge', (), {'travel_time': 1})).travel_time
                occupied_edges[edge_key] = (agv.id, i + travel_time)
        
        return agv_states
    
    def _resolve_conflicts(self, agv: AGV, path: List[int], 
                           occupied_nodes: Set[int], occupied_edges: Dict) -> List[int]:
        """Resolve conflicts by waiting and rerouting"""
        resolved_path = path.copy()
        
        # Check for node conflicts
        for i, node in enumerate(path):
            if node in occupied_nodes and node != agv.origin:
                # Try to find alternative path
                alt_paths = self.path_planner.find_k_shortest_paths(agv.origin, agv.destination, k=3)
                
                for alt_path, alt_cost in alt_paths:
                    if alt_path != path and self._path_is_conflict_free(alt_path, occupied_nodes):
                        resolved_path = alt_path
                        break
                
                # If no alternative found, add waiting time
                if resolved_path == path:
                    print(f"AGV {agv.id} will wait at node {node}")
        
        return resolved_path
    
    def _path_is_conflict_free(self, path: List[int], occupied_nodes: Set[int]) -> bool:
        """Check if path is conflict-free"""
        return not any(node in occupied_nodes for node in path if node != path[0])
    
    def simulate_execution(self, agv_states: Dict[int, AGVState], time_horizon: int) -> Dict[int, List[Tuple[int, int]]]:
        """Simulate AGV execution over time"""
        trajectories = {}
        
        for agv_id, state in agv_states.items():
            trajectory = [(0, state.current_node)]  # (time, node)
            current_time = 0
            current_node = state.current_node
            remaining_path = state.path.copy()
            
            while remaining_path and current_time < time_horizon:
                next_node = remaining_path.pop(0)
                current_time += 1
                current_node = next_node
                trajectory.append((current_time, current_node))
            
            trajectories[agv_id] = trajectory
        
        return trajectories

# Test priority scheduler
priority_scheduler = PriorityScheduler(nodes, edges, path_planner, conflict_detector)

print("=== PRIORITY SCHEDULER TEST ===")
agv_states = priority_scheduler.schedule_agvs(agvs, TIME_HORIZON)

print(f"Scheduled {len(agv_states)} AGVs")
for agv_id, state in agv_states.items():
    print(f"AGV {agv_id}: {state.current_node} → {state.target_node}, path length: {len(state.path) + 1}")

## Step 4 — Greedy with look-ahead heuristic

This heuristic adds **look-ahead capability** to make better local decisions:

- **Look-ahead window**: Evaluate future consequences
- **Congestion prediction**: Anticipate busy areas
- **Dynamic priority adjustment**: Adapt priorities based on context

In [ ]:
class LookAheadScheduler:
    """Greedy scheduler with look-ahead capability"""
    
    def __init__(self, nodes: List[Node], edges: List[Edge], path_planner: PathPlanner,
                 conflict_detector: ConflictDetector, look_ahead: int = 3):
        self.nodes = nodes
        self.edges = edges
        self.path_planner = path_planner
        self.conflict_detector = conflict_detector
        self.look_ahead = look_ahead
        self.node_lookup = {n.id: n for n in nodes}
    
    def predict_congestion(self, agv_states: Dict[int, AGVState], time_horizon: int) -> Dict[int, float]:
        """Predict congestion levels for each node"""
        node_congestion = defaultdict(int)
        
        for state in agv_states.values():
            if state.completed:
                continue
            
            # Count AGVs that will pass through each node
            node_congestion[state.current_node] += 1
            for node in state.path[:self.look_ahead]:
                node_congestion[node] += 1
        
        # Normalize congestion scores
        max_congestion = max(node_congestion.values()) if node_congestion else 1
        return {node: congestion / max_congestion for node, congestion in node_congestion.items()}
    
    def calculate_path_cost_with_congestion(self, path: List[int], 
                                          congestion: Dict[int, float]) -> float:
        """Calculate path cost considering congestion"""
        base_cost = 0
        congestion_penalty = 0
        
        for i in range(len(path) - 1):
            from_node = path[i]
            to_node = path[i + 1]
            
            # Base travel time
            edge_key = (from_node, to_node)
            if edge_key in self.path_planner.edge_lookup:
                base_cost += self.path_planner.edge_lookup[edge_key].travel_time
            else:
                base_cost += 1
            
            # Congestion penalty
            congestion_penalty += congestion.get(to_node, 0) * 2  # Weight congestion
        
        return base_cost + congestion_penalty
    
    def schedule_agvs(self, agvs: List[AGV], time_horizon: int) -> Dict[int, AGVState]:
        """Schedule AGVs using look-ahead greedy approach"""
        agv_states = {}
        congestion = {}  # Will be updated iteratively
        
        # Sort by priority initially
        remaining_agvs = agvs.copy()
        
        while remaining_agvs:
            # Update congestion prediction
            congestion = self.predict_congestion(agv_states, time_horizon)
            
            # Select best AGV to schedule (consider priority and congestion impact)
            best_agv = None
            best_score = float('inf')
            best_path = None
            
            for agv in remaining_agvs:
                # Find multiple path options
                path_options = self.path_planner.find_k_shortest_paths(agv.origin, agv.destination, k=3)
                
                for path, base_cost in path_options:
                    # Calculate total cost with congestion
                    total_cost = self.calculate_path_cost_with_congestion(path, congestion)
                    
                    # Priority-based scoring
                    priority_score = agv.priority * 0.5
                    final_score = total_cost + priority_score
                    
                    if final_score < best_score:
                        best_score = final_score
                        best_agv = agv
                        best_path = path
            
            if best_agv is None:
                break
            
            # Schedule the best AGV
            state = AGVState(
                agv_id=best_agv.id,
                current_node=best_agv.origin,
                target_node=best_agv.destination,
                current_time=0,
                path=best_path[1:],  # Exclude current node
                waiting=False,
                completed=False
            )
            
            agv_states[best_agv.id] = state
            remaining_agvs.remove(best_agv)
        
        return agv_states

# Test look-ahead scheduler
look_ahead_scheduler = LookAheadScheduler(nodes, edges, path_planner, conflict_detector, look_ahead=3)

print("=== LOOK-AHEAD SCHEDULER TEST ===")
la_agv_states = look_ahead_scheduler.schedule_agvs(agvs, TIME_HORIZON)

print(f"Scheduled {len(la_agv_states)} AGVs with look-ahead")
for agv_id, state in la_agv_states.items():
    print(f"AGV {agv_id}: {state.current_node} → {state.target_node}, path: {state.current_node, *state.path}")

## Step 5 — Performance comparison

Let's compare the heuristics against each other and theoretical optimal:

- **Solution quality**: Total completion time
- **Computation time**: Algorithm efficiency
- **Conflict resolution**: Effectiveness of conflict handling
- **Scalability**: Performance on larger instances

In [ ]:
def compare_heuristics(agvs: List[AGV], time_horizon: int) -> pd.DataFrame:
    """Compare different heuristic approaches"""
    results = []
    
    # Test Priority Scheduler
    print("Testing Priority Scheduler...")
    start_time = time.time()
    priority_states = priority_scheduler.schedule_agvs(agvs, time_horizon)
    priority_time = time.time() - start_time
    
    priority_trajectories = priority_scheduler.simulate_execution(priority_states, time_horizon)
    priority_completion = sum(tra[-1][0] for tra in priority_trajectories.values() if tra)
    priority_conflicts = conflict_detector.detect_all_conflicts(list(priority_states.values()))
    
    results.append({
        'Algorithm': 'Priority-Based',
        'Computation_Time': priority_time,
        'Total_Completion': priority_completion,
        'Node_Conflicts': len(priority_conflicts['node_conflicts']),
        'Edge_Conflicts': len(priority_conflicts['edge_conflicts']),
        'AGVs_Scheduled': len(priority_states)
    })
    
    # Test Look-Ahead Scheduler
    print("Testing Look-Ahead Scheduler...")
    start_time = time.time()
    la_states = look_ahead_scheduler.schedule_agvs(agvs, time_horizon)
    la_time = time.time() - start_time
    
    la_trajectories = priority_scheduler.simulate_execution(la_states, time_horizon)
    la_completion = sum(tra[-1][0] for tra in la_trajectories.values() if tra)
    la_conflicts = conflict_detector.detect_all_conflicts(list(la_states.values()))
    
    results.append({
        'Algorithm': 'Look-Ahead Greedy',
        'Computation_Time': la_time,
        'Total_Completion': la_completion,
        'Node_Conflicts': len(la_conflicts['node_conflicts']),
        'Edge_Conflicts': len(la_conflicts['edge_conflicts']),
        'AGVs_Scheduled': len(la_states)
    })
    
    # Test Simple FCFS (First-Come-First-Served) baseline
    print("Testing FCFS Baseline...")
    start_time = time.time()
    fcfs_states = schedule_fcfs(agvs, path_planner, time_horizon)
    fcfs_time = time.time() - start_time
    
    fcfs_trajectories = priority_scheduler.simulate_execution(fcfs_states, time_horizon)
    fcfs_completion = sum(tra[-1][0] for tra in fcfs_trajectories.values() if tra)
    fcfs_conflicts = conflict_detector.detect_all_conflicts(list(fcfs_states.values()))
    
    results.append({
        'Algorithm': 'FCFS Baseline',
        'Computation_Time': fcfs_time,
        'Total_Completion': fcfs_completion,
        'Node_Conflicts': len(fcfs_conflicts['node_conflicts']),
        'Edge_Conflicts': len(fcfs_conflicts['edge_conflicts']),
        'AGVs_Scheduled': len(fcfs_states)
    })
    
    return pd.DataFrame(results)

def schedule_fcfs(agvs: List[AGV], path_planner: PathPlanner, time_horizon: int) -> Dict[int, AGVState]:
    """Simple First-Come-First-Served scheduler"""
    agv_states = {}
    
    for agv in agvs:  # Process in order (FCFS)
        path, _ = path_planner.dijkstra(agv.origin, agv.destination)
        
        if path:
            state = AGVState(
                agv_id=agv.id,
                current_node=agv.origin,
                target_node=agv.destination,
                current_time=0,
                path=path[1:],
                waiting=False,
                completed=False
            )
            agv_states[agv.id] = state
    
    return agv_states

# Compare heuristics
comparison_results = compare_heuristics(agvs, TIME_HORIZON)
display(comparison_results)

print("\n=== PERFORMANCE ANALYSIS ===")
best_completion = comparison_results['Total_Completion'].min()
best_algorithm = comparison_results.loc[comparison_results['Total_Completion'].idxmin(), 'Algorithm']

print(f"Best completion time: {best_completion:.1f} ({best_algorithm})")
print(f"Fastest algorithm: {comparison_results.loc[comparison_results['Computation_Time'].idxmin(), 'Algorithm']} "
      f"({comparison_results['Computation_Time'].min():.3f}s)")

## Step 6 — Comprehensive visualization

Visualize the heuristic performance and solution characteristics:

In [ ]:
def visualize_heuristic_comparison(results_df: pd.DataFrame, priority_states: Dict, 
                                  la_states: Dict, fcfs_states: Dict):
    """Create comprehensive visualization of heuristic comparison"""
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # Colors for consistency
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#F7B731']
    
    # ----------------------------
    # Plot 1: Completion time comparison
    # ----------------------------
    ax1 = axes[0, 0]
    bars = ax1.bar(results_df['Algorithm'], results_df['Total_Completion'], 
                   color=colors[:len(results_df)], alpha=0.8, edgecolor='black', linewidth=1)
    
    ax1.set_title('Total Completion Time Comparison')
    ax1.set_ylabel('Completion Time')
    ax1.tick_params(axis='x', rotation=45)
    ax1.grid(True, alpha=0.3)
    
    # Add value labels
    for bar, value in zip(bars, results_df['Total_Completion']):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{value:.1f}', ha='center', va='bottom', fontweight='bold')
    
    # ----------------------------
    # Plot 2: Computation time comparison
    # ----------------------------
    ax2 = axes[0, 1]
    bars = ax2.bar(results_df['Algorithm'], results_df['Computation_Time'], 
                   color=colors[:len(results_df)], alpha=0.8, edgecolor='black', linewidth=1)
    
    ax2.set_title('Computation Time Comparison')
    ax2.set_ylabel('Time (seconds)')
    ax2.tick_params(axis='x', rotation=45)
    ax2.grid(True, alpha=0.3)
    
    for bar, value in zip(bars, results_df['Computation_Time']):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{value:.3f}', ha='center', va='bottom', fontweight='bold')
    
    # ----------------------------
    # Plot 3: Conflict comparison
    # ----------------------------
    ax3 = axes[0, 2]
    x = np.arange(len(results_df))
    width = 0.35
    
    ax3.bar(x - width/2, results_df['Node_Conflicts'], width, label='Node Conflicts', 
            color='#E63946', alpha=0.8)
    ax3.bar(x + width/2, results_df['Edge_Conflicts'], width, label='Edge Conflicts', 
            color='#A8DADC', alpha=0.8)
    
    ax3.set_title('Conflict Comparison')
    ax3.set_ylabel('Number of Conflicts')
    ax3.set_xticks(x)
    ax3.set_xticklabels(results_df['Algorithm'], rotation=45)
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # ----------------------------
    # Plot 4: Priority scheduler routes
    # ----------------------------
    ax4 = axes[1, 0]
    visualize_routes(ax4, priority_states, 'Priority-Based Scheduler')
    
    # ----------------------------
    # Plot 5: Look-ahead scheduler routes
    # ----------------------------
    ax5 = axes[1, 1]
    visualize_routes(ax5, la_states, 'Look-Ahead Scheduler')
    
    # ----------------------------
    # Plot 6: Performance radar chart
    # ----------------------------
    ax6 = axes[1, 2]
    create_performance_radar(ax6, results_df)
    
    plt.tight_layout()
    plt.show()

def visualize_routes(ax, agv_states: Dict[int, AGVState], title: str):
    """Visualize AGV routes for a specific scheduler"""
    # Draw terminal layout
    for edge in edges:
        from_node = node_lookup[edge.from_node]
        to_node = node_lookup[edge.to_node]
        ax.plot([from_node.x, to_node.x], [from_node.y, to_node.y], 
               'k-', alpha=0.2, linewidth=1)
    
    # Draw nodes
    for node in nodes:
        ax.scatter(node.x, node.y, s=100, c='lightgray', edgecolors='black')
        ax.text(node.x, node.y, str(node.id), ha='center', va='center', fontsize=8)
    
    # Draw AGV routes
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#F7B731', '#5F27CD', '#00D2D3', '#FF9FF3']
    
    for i, (agv_id, state) in enumerate(agv_states.items()):
        path = [state.current_node] + state.path
        if len(path) > 1:
            x_coords = [node_lookup[node].x for node in path]
            y_coords = [node_lookup[node].y for node in path]
            
            ax.plot(x_coords, y_coords, 'o-', color=colors[i % len(colors)], 
                   linewidth=2, markersize=4, label=f'AGV {agv_id}')
    
    ax.set_title(title)
    ax.set_xlabel('X Coordinate')
    ax.set_ylabel('Y Coordinate')
    ax.legend(fontsize=8, loc='upper right')
    ax.set_aspect('equal')

def create_performance_radar(ax, results_df: pd.DataFrame):
    """Create radar chart for performance comparison"""
    # Normalize metrics for radar chart
    metrics = ['Computation_Time', 'Total_Completion', 'AGVs_Scheduled']
    
    # Inverse computation time and completion time (lower is better)
    normalized_data = results_df[metrics].copy()
    normalized_data['Computation_Time'] = 1 / (normalized_data['Computation_Time'] + 0.001)
    normalized_data['Total_Completion'] = 1 / normalized_data['Total_Completion']
    
    # Scale to 0-1 range
    for metric in metrics:
        max_val = normalized_data[metric].max()
        min_val = normalized_data[metric].min()
        if max_val > min_val:
            normalized_data[metric] = (normalized_data[metric] - min_val) / (max_val - min_val)
        else:
            normalized_data[metric] = 1.0
    
    # Create radar chart
    angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
    angles += angles[:1]  # Complete the circle
    
    ax = plt.subplot(111, projection='polar')
    
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
    
    for i, (idx, row) in enumerate(normalized_data.iterrows()):
        values = row.tolist()
        values += values[:1]  # Complete the circle
        
        ax.plot(angles, values, 'o-', linewidth=2, label=results_df.loc[idx, 'Algorithm'],
               color=colors[i % len(colors)])
        ax.fill(angles, values, alpha=0.25, color=colors[i % len(colors)])
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(['Speed', 'Quality', 'Coverage'])
    ax.set_ylim(0, 1)
    ax.set_title('Performance Radar Chart')
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))

# Create visualizations
visualize_heuristic_comparison(comparison_results, priority_states, la_states, fcfs_states)

## Step 7 — Scalability analysis

Test how heuristics perform on **larger problem instances**:

In [ ]:
def scalability_test():
    """Test scalability with increasing problem sizes"""
    print("=== SCALABILITY ANALYSIS ===")
    
    # Test different problem sizes
    test_sizes = [4, 6, 8, 10, 12]  # Number of AGVs
    scalability_results = []
    
    for num_agvs in test_sizes:
        print(f"\nTesting with {num_agvs} AGVs...")
        
        # Create test instance
        test_agvs = agvs[:num_agvs] if num_agvs <= len(agvs) else create_random_agvs(num_agvs)
        
        # Test priority scheduler
        start_time = time.time()
        priority_states = priority_scheduler.schedule_agvs(test_agvs, TIME_HORIZON)
        priority_time = time.time() - start_time
        
        # Test look-ahead scheduler
        start_time = time.time()
        la_states = look_ahead_scheduler.schedule_agvs(test_agvs, TIME_HORIZON)
        la_time = time.time() - start_time
        
        scalability_results.append({
            'Num_AGVs': num_agvs,
            'Priority_Time': priority_time,
            'LookAhead_Time': la_time,
            'Priority_Scheduled': len(priority_states),
            'LookAhead_Scheduled': len(la_states)
        })
        
        print(f"  Priority: {priority_time:.3f}s, {len(priority_states)} scheduled")
        print(f"  Look-ahead: {la_time:.3f}s, {len(la_states)} scheduled")
    
    # Create scalability DataFrame
    scalability_df = pd.DataFrame(scalability_results)
    display(scalability_df)
    
    # Visualize scalability
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Computation time vs problem size
    ax1.plot(scalability_df['Num_AGVs'], scalability_df['Priority_Time'], 
            'o-', linewidth=2, markersize=8, color='#FF6B6B', label='Priority-Based')
    ax1.plot(scalability_df['Num_AGVs'], scalability_df['LookAhead_Time'], 
            's-', linewidth=2, markersize=8, color='#4ECDC4', label='Look-Ahead')
    
    ax1.set_title('Computation Time Scalability')
    ax1.set_xlabel('Number of AGVs')
    ax1.set_ylabel('Computation Time (seconds)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Success rate vs problem size
    ax2.plot(scalability_df['Num_AGVs'], scalability_df['Priority_Scheduled'], 
            'o-', linewidth=2, markersize=8, color='#FF6B6B', label='Priority-Based')
    ax2.plot(scalability_df['Num_AGVs'], scalability_df['LookAhead_Scheduled'], 
            's-', linewidth=2, markersize=8, color='#4ECDC4', label='Look-Ahead')
    
    ax2.set_title('Scheduling Success Rate')
    ax2.set_xlabel('Number of AGVs')
    ax2.set_ylabel('Number of AGVs Successfully Scheduled')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return scalability_df

def create_random_agvs(num_agvs: int) -> List[AGV]:
    """Create random AGVs for testing"""
    random_agvs = []
    node_ids = [n.id for n in nodes]
    
    for i in range(num_agvs):
        origin = random.choice(node_ids)
        destination = random.choice([n for n in node_ids if n != origin])
        priority = random.randint(1, 3)
        speed = random.uniform(0.8, 1.2)
        
        random_agvs.append(AGV(i + 1, origin, destination, priority, speed))
    
    return random_agvs

# Run scalability analysis
scalability_results = scalability_test()

## Summary and key insights

This Tier-2 notebook demonstrates **heuristic approaches** for AGV traffic management:

### Heuristic strengths
- **Computational efficiency**: Fast solutions for real-time operations
- **Scalability**: Handle larger instances than exact methods
- **Flexibility**: Easy to adapt to different constraints
- **Practical applicability**: Suitable for dynamic environments

### Algorithm comparison
- **Priority-based**: Simple, fast, respects AGV importance
- **Look-ahead**: Better solution quality, considers future impact
- **FCFS baseline**: Basic reference for comparison

### Trade-offs identified
- **Speed vs quality**: Faster algorithms may sacrifice optimality
- **Complexity vs maintainability**: More sophisticated heuristics harder to debug
- **Determinism vs adaptability**: Fixed rules vs dynamic adjustment

### Practical considerations
- **Real-time requirements**: Heuristics essential for live operations
- **Dynamic environments**: Need online replanning capabilities
- **Integration requirements**: Interface with terminal management systems

### Next steps
- Explore metaheuristics (GA, PSO) in Tier-3 for better solutions
- Investigate reinforcement learning for adaptive decision making in Tier-4
- Consider hybrid approaches combining multiple heuristics